# Quickstart

This notebook walks through one short example from start to finish. We load daily temperature data, cut it down to California in summer, convert it from Kelvin to Celsius, average it over the region and save the result to a NetCDF file.

If you want more options or more detail on any of these steps, see [`subsetting-and-exporting.ipynb`](subsetting-and-exporting.ipynb).

Data associated with this repository are subject to additional [terms of data access](https://carbonplan.github.io/srm-downscaling/terms-of-data-access.html). If you come across a term you don't know, check the [glossary](https://github.com/carbonplan/sai-downscaling-data-utils/blob/main/GLOSSARY.md).

## Setup

Follow the [installation instructions](https://github.com/carbonplan/sai-downscaling-data-utils#installation) in the README, then launch JupyterLab with `pixi run jupyter lab`. The cell below loads helper functions from the [`scripts/`](../scripts/README.md) folder, so open this notebook from inside the cloned repository.

In [1]:
import sys
from pathlib import Path

import numpy as np

# The helper functions live in the repository's scripts/ folder.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "data_access.py").exists()), None)
if repo is None:
    raise RuntimeError("Open this notebook from inside the sai-downscaling-data-utils repository.")
sys.path.insert(0, str(repo / "scripts"))

from data_access import VARIABLES, load_downscaling_store, members_for, output_filename, scenarios_for  # noqa: E402


# Convert Kelvin to Celsius; adapt this function for other unit conversions
def kelvin_to_celsius(da):
    da_celsius = da - 273.15
    da_celsius.attrs = da.attrs.copy()
    da_celsius.attrs["units"] = "°C"
    return da_celsius

## 1. Choose your data, region and season

Change the values in the cell below to pick your own data. The example uses summer (June to August) temperatures for California from 2015 to 2020.

In [2]:
# ===== CUSTOMIZE ALL PARAMETERS FOR YOUR WORKFLOW =====

# 1. Scenario selection
workflow_scenario = "ssp245"  # see scenarios_for(workflow_gcm)
workflow_variable = "tas"  # any of VARIABLES
workflow_gcm = "CESM2-WACCM6"  # "CESM2-WACCM6" or "UKESM1-1-LL"
workflow_method = "bcsd"  # "bcsd" or "qdmsd"
workflow_member = None  # None takes the pinned default; see members_for(...)
workflow_product = "downscaled"  # or "debiased_coarse" (see Section 8 of subsetting-and-exporting.ipynb)

# 2. Spatial subsetting (bounding box for California)
workflow_bbox = {
    "lon_min": -124.5,
    "lon_max": -114.0,
    "lat_min": 32.5,
    "lat_max": 42.0,
}

# 3. Temporal subsetting
workflow_start_date = "2015-01-01"
workflow_end_date = "2020-12-31"
workflow_months = [6, 7, 8]  # Summer months (JJA)

# 4. Output
workflow_label = "california-summer"  # the scenario, GCM, member and dates are added automatically
output_dir = Path("./exports")  # where the file is written; created if it does not exist

# =======================================================

## 2. Run the workflow

The cell below loads the data, subsets it, converts the units, takes the regional average and saves the file. The data isn't actually downloaded until the file is written, and only the parts covering your region and years are read.

In [3]:
# Run the workflow: load, subset, convert, average and export
output_dir.mkdir(exist_ok=True)

# Step 1: Load scenario
print("Step 1: Loading dataset...")
ds_workflow = load_downscaling_store(
    scenario=workflow_scenario, variable=workflow_variable,
    gcm=workflow_gcm, method=workflow_method, member=workflow_member,
    product=workflow_product,
)
print(f"  Loaded dataset with shape: {dict(ds_workflow.sizes)}")

# Step 2: Spatial subset (bounding box)
print("\nStep 2: Applying spatial subset...")
ds_workflow = ds_workflow.sel(
    lat=slice(workflow_bbox["lat_min"], workflow_bbox["lat_max"]),
    lon=slice(workflow_bbox["lon_min"], workflow_bbox["lon_max"]),
)
print(f"  Spatial subset: {dict(ds_workflow.sizes)}")

# Step 3: Temporal subset
print("\nStep 3: Applying temporal subset...")
ds_workflow = ds_workflow.sel(time=slice(workflow_start_date, workflow_end_date))
ds_workflow = ds_workflow.sel(time=ds_workflow.time.dt.month.isin(workflow_months))
print(f"  Temporal subset: {ds_workflow.sizes['time']} time steps")

# Step 4: Unit conversion
print("\nStep 4: Converting units...")
ds_workflow["tas_celsius"] = kelvin_to_celsius(ds_workflow[workflow_variable])
print("  Added tas_celsius variable")

# Step 5: Regional mean and export
print("\nStep 5: Calculating regional mean and exporting...")
weights_workflow = np.cos(np.deg2rad(ds_workflow.lat))
ds_workflow_mean = (
    ds_workflow[["tas_celsius"]].drop_attrs().weighted(weights_workflow).mean(dim=["lat", "lon"])
)

workflow_output_path = output_dir / output_filename(
    workflow_scenario, workflow_variable, workflow_start_date, workflow_end_date,
    gcm=workflow_gcm, method=workflow_method, member=workflow_member,
    product=workflow_product, label=workflow_label, months=workflow_months,
)
ds_workflow_mean.to_netcdf(workflow_output_path)
print(f"  Exported to: {workflow_output_path}")
print(f"  File size: {workflow_output_path.stat().st_size / 1024:.1f} KB")

print("\nComplete workflow finished successfully!")
ds_workflow_mean

Step 1: Loading dataset...


  Loaded dataset with shape: {'time': 31046, 'lat': 721, 'lon': 1440}

Step 2: Applying spatial subset...
  Spatial subset: {'time': 31046, 'lat': 39, 'lon': 43}

Step 3: Applying temporal subset...
  Temporal subset: 552 time steps

Step 4: Converting units...
  Added tas_celsius variable

Step 5: Calculating regional mean and exporting...


  Exported to: exports/california-summer_CESM2-WACCM6_bcsd_ssp245_003_tas_2015-2020_JJA.nc
  File size: 4.6 KB

Complete workflow finished successfully!


<xarray.Dataset> Size: 7kB
Dimensions:      (time: 552)
Coordinates:
  * time         (time) datetime64[ns] 4kB 2015-06-01 2015-06-02 ... 2020-08-31
Data variables:
    tas_celsius  (time) float32 2kB dask.array<chunksize=(313,), meta=np.ndarray>

## Next steps

[`subsetting-and-exporting.ipynb`](subsetting-and-exporting.ipynb) goes into each of these steps in more detail:

- **Choosing data** (Section 2): other GCMs, methods, scenarios and ensemble members, and how to check how big a request is before you run it.
- **Picking a region** (Section 3): a single point or a country outline instead of a box.
- **Picking a time period** (Section 4): seasons, climatologies and annual means.
- **Quality flags** (Section 5.5): how to mask out data with known issues.
- **Saving files** (Section 6): Zarr, CSV and Parquet as well as NetCDF.
- **Bias-corrected data** (Section 8): the data at the GCM's resolution, before it is downscaled to 0.25°.

If you just want a file and don't need a notebook, `./scripts/download.sh` downloads data from the command line.